# A04 — Browser gerenciado: mouse, teclado e capturas

![Arquitetura do exercício](assets/browser.png)

## O que você vai aprender

Você cria a role IAM e o Browser pelo SDK nas células, inicia uma sessão e executa ações de mouse, teclado e captura de tela. AgentCore Browser hospeda o navegador na AWS. Este exemplo usa boto3 diretamente e não exige LLM, Nova Act ou credencial externa.

## Pré-requisitos e sequência

Use uma conta AWS com sessão autorizada em `us-west-2`, Python 3.12 e kernel próprio para este apêndice.
As células instalam as versões validadas e criam os recursos durante a execução. Execute de cima para baixo, incluindo a limpeza.
Uma conta basta; as permissões variam conforme o evento. Estes exemplos são independentes das aulas Santander.

## Como interpretar o resultado

As **28 ações** devem devolver `SUCCESS` e as duas capturas devem conter imagens PNG. Isso comprova a execução das APIs; não comprova um fluxo de negócio em uma página. A limpeza encerra a sessão e remove apenas os recursos desta execução.

## Origem e adaptações

Derivado do [notebook oficial](https://github.com/awslabs/agentcore-samples/blob/0d46da96306fa692c9887d0461252bf5aeeee6c3/06-workshops/05-AgentCore-tools/02-Agent-Core-browser-tool/14-browser-os-actions/browser-os-actions-boto3.ipynb), revisão `0d46da96306f`.
Foram adicionados explicações, validações e dependências explícitas; os caminhos dos arquivos foram organizados neste apêndice.
O exemplo usa a cadeia padrão de credenciais; não salve credenciais no notebook.
As evidências e diferenças constam no [relatório de validação](VALIDACAO.md).

## Conferir a sessão AWS


In [ ]:
import boto3, os
session = boto3.Session()
REGION = session.region_name or os.environ.get("AWS_DEFAULT_REGION")
assert REGION == "us-west-2", "Este apêndice foi validado em us-west-2; configure a região antes de continuar."
identity = session.client("sts").get_caller_identity()
print({"conta": identity["Account"], "regiao": REGION, "identidade": identity["Arn"]})
# Credenciais vêm da cadeia padrão do boto3: role do ambiente, perfil ou sessão temporária.
# Nenhuma chave, senha ou token deve ser salvo neste notebook.


In [ ]:
%pip install "boto3==1.43.124" "requests==2.32.5"
import boto3
print(f"boto3: {boto3.__version__}")

### 2. Load AWS credentials

Use an authorized AWS session through the default boto3 credentials chain (environment role, profile, or temporary session). Do not save credentials in notebook cells.

In [ ]:
import boto3, os
session = boto3.Session()
REGION = session.region_name or os.environ.get("AWS_DEFAULT_REGION")
assert REGION == "us-west-2", "Este apêndice foi validado em us-west-2; configure a região antes de continuar."
identity = session.client("sts").get_caller_identity()
print({"conta": identity["Account"], "regiao": REGION, "identidade": identity["Arn"]})
# Credenciais vêm da cadeia padrão do boto3: role do ambiente, perfil ou sessão temporária.
# Nenhuma chave, senha ou token deve ser salvo neste notebook.


### 3. Setup boto3 clients

In [ ]:
import time

REGION = "us-west-2"
import uuid
RUN_SUFFIX = uuid.uuid4().hex[:10]
BROWSER_NAME = "appendix_browser_" + RUN_SUFFIX

# Control plane — create/delete browsers
control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

# Data plane — sessions and invoke
runtime_client = boto3.client("bedrock-agentcore", region_name=REGION)

print(f"Clients ready — region: {REGION}")

### 3.1 Create IAM Role

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import browser_helpers
SAMPLE_ROLE_NAME = "AppendixBrowserRole_" + RUN_SUFFIX
browser_helpers.POLICY_NAME = "AppendixBrowserPolicy_" + RUN_SUFFIX
execution_role_arn = browser_helpers.create_agentcore_execution_role(SAMPLE_ROLE_NAME)
assert execution_role_arn, "A criação da role falhou; não continue."
print("Aguardando propagação do IAM...")
time.sleep(10)


### 4. Create the AgentCore Browser

In [ ]:
created = control_client.create_browser(
    name=BROWSER_NAME,
    executionRoleArn=execution_role_arn,
    networkConfiguration={"networkMode": "PUBLIC"},
)

browser_id = created["browserId"]
print(f"Browser created: {browser_id}")

### 5. Start a browser session

In [ ]:
session = runtime_client.start_browser_session(
    browserIdentifier=browser_id,
    name=BROWSER_NAME,
    sessionTimeoutSeconds=3600,
    viewPort={"width": 1920, "height": 1080},
)

session_id = session["sessionId"]
print(f"Session started: {session_id}")

print("Waiting 3s for session init...")
time.sleep(3)

### Helper — invoke wrapper

Small wrapper to keep the action cells clean.

In [ ]:
action_results = []
def invoke(action: dict) -> dict:
    resp = runtime_client.invoke_browser(browserIdentifier=browser_id, sessionId=session_id, action=action)
    result = resp["result"]
    assert result and all(item.get("status") == "SUCCESS" for item in result.values()), result
    action_results.append(result)
    return result
print("invoke() valida cada retorno SUCCESS.")


### 6. Mouse Actions

In [ ]:
print("── Mouse Click Actions ──\n")

# Left click
r = invoke({"mouseClick": {"x": 600, "y": 370, "button": "LEFT"}})
print(f"Left click: {r}")

# Double click
r = invoke({"mouseClick": {"x": 500, "y": 300, "button": "LEFT", "clickCount": 2}})
print(f"Double click: {r}")

# Right click
r = invoke({"mouseClick": {"x": 200, "y": 400, "button": "RIGHT", "clickCount": 1}})
print(f"Right click: {r}")

# Middle click
r = invoke({"mouseClick": {"x": 960, "y": 540, "button": "MIDDLE", "clickCount": 1}})
print(f"Middle click: {r}")

# Multi click
r = invoke({"mouseClick": {"x": 500, "y": 300, "button": "LEFT", "clickCount": 10}})
print(f"Multi click (10x): {r}")

In [ ]:
print("── Mouse Move Actions ──\n")

r = invoke({"mouseMove": {"x": 800, "y": 600}})
print(f"Move to (800,600): {r}")

r = invoke({"mouseMove": {"x": 1, "y": 1}})
print(f"Move to (1,1): {r}")

In [ ]:
print("── Mouse Drag Actions ──\n")

r = invoke(
    {
        "mouseDrag": {
            "startX": 1,
            "startY": 1,
            "endX": 705,
            "endY": 180,
            "button": "LEFT",
        }
    }
)
print(f"Drag (1,1)->(705,180): {r}")

r = invoke(
    {
        "mouseDrag": {
            "startX": 1,
            "startY": 1,
            "endX": 370,
            "endY": 330,
            "button": "LEFT",
        }
    }
)
print(f"Drag (1,1)->(370,330): {r}")

r = invoke(
    {
        "mouseDrag": {
            "startX": 500,
            "startY": 300,
            "endX": 100,
            "endY": 200,
            "button": "MIDDLE",
        }
    }
)
print(f"Drag middle button: {r}")

### 7. Scroll Actions

In [ ]:
print("── Scroll Actions ──\n")

r = invoke({"mouseScroll": {"x": 800, "y": 600, "deltaX": 0, "deltaY": -500}})
print(f"Scroll up: {r}")

r = invoke({"mouseScroll": {"x": 500, "y": 300, "deltaX": 300, "deltaY": 0}})
print(f"Scroll right: {r}")

r = invoke({"mouseScroll": {"x": 500, "y": 300, "deltaX": -100, "deltaY": -200}})
print(f"Scroll diagonal: {r}")

r = invoke({"mouseScroll": {"x": 500, "y": 300, "deltaX": 1000, "deltaY": 1000}})
print(f"Scroll large: {r}")

### 8. Keyboard Actions

In [ ]:
print("── Key Type Actions ──\n")

r = invoke({"keyType": {"text": "Hello World"}})
print(f"Type 'Hello World': {r}")

r = invoke({"keyType": {"text": "user@example.com!#$%^&*()"}})
print(f"Type special chars: {r}")

r = invoke({"keyType": {"text": "https://www.example.com"}})
print(f"Type URL: {r}")

r = invoke({"keyType": {"text": "1" * 10000}})
print(f"Type long string (10k): {r}")

In [ ]:
print("── Key Press Actions ──\n")

r = invoke({"keyPress": {"key": "enter"}})
print(f"Enter: {r}")

r = invoke({"keyPress": {"key": "tab"}})
print(f"Tab: {r}")

r = invoke({"keyPress": {"key": "escape"}})
print(f"Escape: {r}")

r = invoke({"keyPress": {"key": "backspace", "presses": 5}})
print(f"Backspace x5: {r}")

r = invoke({"keyPress": {"key": "ArrowDown", "presses": 100}})
print(f"ArrowDown x100: {r}")

In [ ]:
print("── Key Shortcut Actions ──\n")

r = invoke({"keyShortcut": {"keys": ["ctrl", "s"]}})
print(f"Ctrl+S: {r}")

r = invoke({"keyShortcut": {"keys": ["ctrl", "p"]}})
print(f"Ctrl+P: {r}")

r = invoke({"keyShortcut": {"keys": ["ctrl", "shift", "i"]}})
print(f"Ctrl+Shift+I: {r}")

### 9. Screenshot

In [ ]:
from IPython.display import Image, display
import base64
screenshot_sizes = []
def take_screenshot(fmt="PNG"):
    result = invoke({"screenshot": {"format": fmt}} if fmt else {"screenshot": {}})
    data = result["screenshot"]["data"]
    image_bytes = data if isinstance(data, bytes) else base64.b64decode(data)
    assert image_bytes.startswith(b"\x89PNG\r\n\x1a\n"), "A captura não contém PNG válido."
    screenshot_sizes.append(len(image_bytes))
    print({"status": result["screenshot"]["status"], "bytes": len(image_bytes)})
    display(Image(image_bytes))
take_screenshot("PNG")
take_screenshot(None)
assert len(action_results) == 28
assert len(screenshot_sizes) == 2 and min(screenshot_sizes) > 0
print("VALIDADO: 28 ações SUCCESS e duas imagens PNG.")


### 10. Stop the browser session

In [ ]:
runtime_client.stop_browser_session(browserIdentifier=browser_id, sessionId=session_id)
print(f"Session {session_id} stopped ✓")

### 11. Cleanup (Optional)

Delete the browser and IAM role.

In [ ]:
control_client.delete_browser(browserId=browser_id)
print(f"Browser {browser_id} deleted ✓")

In [ ]:
browser_helpers.delete_agentcore_execution_role(SAMPLE_ROLE_NAME)
iam = boto3.client("iam")
try:
    iam.get_role(RoleName=SAMPLE_ROLE_NAME)
except iam.exceptions.NoSuchEntityException:
    print("Role desta execução removida.")
else:
    raise AssertionError("A role ainda existe; a limpeza não terminou.")
policy_arn = f"arn:aws:iam::{identity['Account']}:policy/{browser_helpers.POLICY_NAME}"
try:
    iam.get_policy(PolicyArn=policy_arn)
except iam.exceptions.NoSuchEntityException:
    print("Policy desta execução removida.")
else:
    raise AssertionError("A policy ainda existe; a limpeza não terminou.")
